# [6.1] SAE Variants - Exercises

SAEs turn activations into sparse feature coordinates, but a sparse code is only useful if you can test what it preserves and what it breaks. In this notebook you build four encoder rules, measure reconstruction and sparsity separately, recover planted dictionary directions, validate a feature on held-out labels, and test decoder-vector steering against a random control.

```yaml
gt_tier: GT-1 with GT-0 planted-feature controls
exercise_id: 6_1_sae_variants
expected_runtime: 60-90 minutes for CPU exercises; several minutes for CUDA Pythia SAE preflight
requires_gpu: true for the released-checkpoint preflight; false for the implementation exercises
```

<details>
<summary>Expected output</summary>

By the end, every local test should print an "All tests ... passed" line, and the final report-backed cell should show a Pythia-70M TopK SAE signature table with held-out reconstruction, permuted-decoder, feature-AUC, sparsity, steering, and VRAM metrics.

</details>

<details>
<summary>Help - how to read this section</summary>

This is the bridge from toy superposition to real sparse-feature tools. Do not treat a feature name or top activation as evidence by itself. The evidence ladder is: exact toy rule, planted-feature recovery, held-out validation, negative control, then a small real-model preflight.

</details>


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter6_sparse_feature_methods"
section = "part1_sae_variants"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_sae_variants.tests as tests


@dataclass(frozen=True)
class ToySuperpositionBatch:
    feature_acts: t.Tensor
    activations: t.Tensor
    dictionary: t.Tensor


@dataclass(frozen=True)
class SAEVariantMetrics:
    name: str
    l0: float
    feature_density_mean: float
    dead_feature_fraction: float
    reconstruction_mse: float


@dataclass(frozen=True)
class DictionaryRecoveryReport:
    mean_best_cosine: float
    recovered_fraction: float
    duplicate_fraction: float
    best_learned_for_true: t.Tensor


@dataclass(frozen=True)
class FeatureAUCReport:
    feature_id: int
    auc: float


@dataclass(frozen=True)
class SteeringComparisonReport:
    baseline_mean: float
    steered_mean: float
    random_mean: float
    steered_delta: float
    random_delta: float
    passes_control: bool

MAIN = __name__ == "__main__"


## 1. Sparse Encoder Rules

You will implement ReLU/L1 soft-thresholding, TopK, Gated, and JumpReLU encoders. All four take pre-activations and decide which feature coordinates count as evidence.

<details>
<summary>Expected output</summary>

On the controlled test row, TopK keeps exactly the two positive winners, the gate closes features whose gate logit is not above threshold, and JumpReLU preserves only values strictly above the jump threshold.

</details>

<details>
<summary>Help - why TopK comes after ReLU</summary>

TopK is a sparsity rule over nonnegative feature evidence. If you select before ReLU, a large negative value can count as one of the kept features, which is not an SAE feature activation.

</details>


In [ ]:
def relu_l1_encode(pre_acts: t.Tensor, *, l1_coefficient: float = 0.0) -> t.Tensor:
    raise NotImplementedError()


def topk_encode(pre_acts: t.Tensor, *, k: int) -> t.Tensor:
    raise NotImplementedError()


def gated_encode(
    pre_acts: t.Tensor,
    gate_logits: t.Tensor,
    *,
    gate_threshold: float = 0.0,
) -> t.Tensor:
    raise NotImplementedError()


def jumprelu_encode(pre_acts: t.Tensor, *, threshold: float) -> t.Tensor:
    raise NotImplementedError()


tests.test_encoder_variants_match_reference_and_sparsity_rules(
    relu_l1_encode,
    topk_encode,
    gated_encode,
    jumprelu_encode,
)


## 2. Reconstruction And Density

A sparse code is not enough. Decode it back to activation space, then report reconstruction MSE, average L0, density, and dead-feature fraction separately.

<details>
<summary>Expected output</summary>

The identity decoder reconstructs exactly, with MSE `0.0`, L0 `1.5`, mean density `0.5`, and no dead features.

</details>

<details>
<summary>Help - reconstruction and sparsity are separate axes</summary>

Dense features can reconstruct well but tell you little about decomposition. Very sparse features can look clean but lose the activation. Keep both axes visible.

</details>


In [ ]:
def decode_features(
    feature_acts: t.Tensor,
    decoder_weight: t.Tensor,
    decoder_bias: t.Tensor | None = None,
) -> t.Tensor:
    raise NotImplementedError()


def feature_density(feature_acts: t.Tensor, threshold: float = 0.0) -> t.Tensor:
    raise NotImplementedError()


def l0(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    raise NotImplementedError()


def dead_feature_fraction(feature_acts: t.Tensor, threshold: float = 0.0) -> float:
    raise NotImplementedError()


def sae_variant_metrics(
    name: str,
    *,
    activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    feature_acts: t.Tensor,
) -> SAEVariantMetrics:
    raise NotImplementedError()


tests.test_decode_and_metrics_match_identity_contract(
    decode_features,
    sae_variant_metrics,
    feature_density,
    l0,
    dead_feature_fraction,
)


## 3. Planted Sparse Features

Now create a toy superposition batch where the true sparse feature activations and dictionary are known. This is your ground-truth sandbox.

<details>
<summary>Expected output</summary>

With the fixed seed, feature activations have shape `(32, 6)`, activations have shape `(32, 3)`, dictionary rows have shape `(6, 3)`, and the dictionary rows are unit norm.

</details>

<details>
<summary>Help - why planted features come before Pythia</summary>

On real Pythia hidden states, there is no ground-truth feature dictionary. The planted setting gives you one place where recovery and duplicate failures are not matters of taste.

</details>


In [ ]:
def make_toy_superposition_batch(
    *,
    batch: int = 256,
    n_features: int = 8,
    d_model: int = 4,
    feature_probability: float = 0.2,
    noise_scale: float = 0.0,
    seed: int = 0,
) -> ToySuperpositionBatch:
    raise NotImplementedError()


def density_is_nondegenerate(
    feature_acts: t.Tensor,
    *,
    min_active_fraction: float = 0.05,
    max_active_fraction: float = 0.95,
) -> bool:
    raise NotImplementedError()


tests.test_toy_superposition_batch_has_planted_sparse_structure(
    make_toy_superposition_batch,
    density_is_nondegenerate,
)


## 4. Dictionary Recovery

A learned decoder can duplicate one true feature and miss another. Measure recovery from the true-feature side, not only from the learned-feature side.

<details>
<summary>Expected output</summary>

The duplicated decoder recovers `2/3` true directions and reports duplicate fraction `1/3`.

</details>

<details>
<summary>Help - duplicates can fool top-example inspection</summary>

Two learned features with similar top examples may both point at the same true direction. The missing true direction is the important failure.

</details>


In [ ]:
def dictionary_recovery_report(
    learned_decoder: t.Tensor,
    true_dictionary: t.Tensor,
    *,
    threshold: float = 0.8,
) -> DictionaryRecoveryReport:
    raise NotImplementedError()


tests.test_dictionary_recovery_detects_duplicates_and_missing_features(
    dictionary_recovery_report,
)


## 5. Held-Out Feature Validation

Top activating examples are hypotheses. A feature starts to earn trust when it predicts held-out labels or behavior better than controls.

<details>
<summary>Expected output</summary>

One feature ranks positives above negatives perfectly and another ranks positives below negatives perfectly; after polarity correction both count as AUC separation `1.0`.

</details>

<details>
<summary>Help - why AUC instead of a threshold</summary>

A single threshold can be cherry-picked. Rank-based AUC asks whether positives are generally scored above negatives, independent of one cutoff.

</details>


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def best_feature_auc(feature_acts: t.Tensor, labels: t.Tensor) -> FeatureAUCReport:
    raise NotImplementedError()


tests.test_best_feature_auc_handles_predictive_and_antipredictive_features(
    roc_auc_binary,
    best_feature_auc,
)


## 6. Decoder-Vector Steering

Finally, test whether adding a decoder vector moves the score you claim it should move, and compare against a random direction.

<details>
<summary>Expected output</summary>

Default steering changes only the final position, all-position steering changes every position, and the target decoder direction beats the random control.

</details>

<details>
<summary>Help - steering is not a feature name</summary>

A steering effect is evidence that a direction can move a chosen score. It is not by itself evidence that the feature has a clean semantic label.

</details>


In [ ]:
def apply_decoder_steering(
    activations: t.Tensor,
    decoder_vectors: t.Tensor,
    feature_ids: t.Tensor | list[int],
    coefficients: t.Tensor | list[float] | float,
    *,
    positions: Literal["all", "last"] = "last",
) -> t.Tensor:
    raise NotImplementedError()


def steering_comparison_report(
    baseline_scores: t.Tensor,
    steered_scores: t.Tensor,
    random_control_scores: t.Tensor,
) -> SteeringComparisonReport:
    raise NotImplementedError()


tests.test_decoder_steering_changes_last_position_and_reports_control(
    apply_decoder_steering,
    steering_comparison_report,
)


## Whole-Notebook Contract

Once all exercises pass, your local implementation should satisfy the same smoke-test contract as `solutions.py`.


In [ ]:
# Uncomment after finishing the exercises to run the whole local contract.
# tests.test_notebook_contract(run_smoke_test)


## Signature Result

The final result is report-backed and uses the committed CUDA evidence. It does not rerun training inside the notebook; rerun `solutions.run_gpu_test(max_vram_gb=24.0)` from Python when you want to refresh the report.

<details>
<summary>Expected output</summary>

You should see a compact table with Pythia-70M, width-256 TopK-16 SAE, held-out reconstruction MSE around `1.134`, zero baseline MSE around `1.375`, permuted decoder MSE around `1.629`, best feature AUC around `0.984`, steering delta around `5.0`, and peak VRAM around `0.391 GB`.

</details>

<details>
<summary>Interpreting the signature result</summary>

The learned decoder improves held-out reconstruction and the permuted decoder fails, so this is not just a random dictionary. The AUC and steering controls are useful diagnostics, but they remain scoped to the generated technical/everyday prompt split and decoder-projection score.

</details>


In [ ]:
import json


def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model", gpu["model_id"]),
        ("train / held-out prompts", f"{gpu['train_prompt_count']} / {gpu['heldout_prompt_count']}"),
        ("SAE", f"width {gpu['sae_width']}, TopK {gpu['sae_k']}, {gpu['training_steps']} steps"),
        ("held-out MSE", round(gpu["heldout_reconstruction_mse"], 4)),
        ("zero baseline MSE", round(gpu["zero_baseline_mse"], 4)),
        ("permuted decoder MSE", round(gpu["permuted_decoder_mse"], 4)),
        ("best feature AUC", round(gpu["best_feature_auc"], 4)),
        ("held-out L0", gpu["heldout_l0"]),
        ("dead feature fraction", round(gpu["heldout_dead_feature_fraction"], 4)),
        ("steer delta / random delta", f"{gpu['decoder_projection_steered_delta']:.3f} / {gpu['decoder_projection_random_delta']:.2e}"),
        ("safe token logit delta", f"{gpu['safe_logit_token']!r}: {gpu['safe_logit_delta']:.3f}"),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["zero", "learned", "permuted"],
    [gpu["zero_baseline_mse"], gpu["heldout_reconstruction_mse"], gpu["permuted_decoder_mse"]],
    color=["#94a3b8", "#2563eb", "#f97316"],
)
axes[0].set_title("Held-out reconstruction")
axes[0].set_ylabel("MSE, lower is better")

axes[1].bar(
    ["everyday", "technical"],
    [gpu["best_feature_negative_mean"], gpu["best_feature_positive_mean"]],
    color=["#94a3b8", "#16a34a"],
)
axes[1].set_title(f"Feature {gpu['best_feature_id']} AUC={gpu['best_feature_auc']:.3f}")
axes[1].set_ylabel("mean activation")

axes[2].bar(
    ["decoder", "random"],
    [gpu["decoder_projection_steered_delta"], gpu["decoder_projection_random_delta"]],
    color=["#7c3aed", "#94a3b8"],
)
axes[2].set_title("Steering control")
axes[2].set_ylabel("projection delta")

fig.tight_layout()
plt.show()


## Limitations

This section does not prove semantic feature interpretation, full-scale SAE training quality, or that TopK is globally better than ReLU/L1, Gated, or JumpReLU variants. The real-model path trains one tiny TopK SAE on final-token Pythia-70M hidden states for generated safe topic prompts. Later sections use released SAE artifacts, dashboards, transcoders, attribution graphs, and stronger causal feature claims.
